# Quick results: Version 1 proof of concept

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/divyansh070/btp_ideas_implementation/blob/main/quick_results/quick_results_colab.ipynb)

Runtime → Change runtime type → **T4 GPU**. Run all cells. Data and outputs are cached on Google Drive, so a restarted session skips finished steps. See `quick_results/README.md`.

In [ ]:
USE_HUST = True          # +HUST: 77 LFP cells, 1.39 GB download (set False for SNL/CALCE/HNEI only)
RUN_E4 = False          # released PBT zero-shot; needs Colab secret HF_TOKEN with approved Llama-3.1 access
ALLOW_GATE_FAIL = False # True = train even if a validation-gate check fails (the report still shows FAIL)
!nvidia-smi -L

## Clone

In [ ]:
import os
if not os.path.exists("/content/btp"):
    !git clone -q https://github.com/divyansh070/btp_ideas_implementation /content/btp
%cd /content/btp/quick_results
!git pull -q --ff-only
!git log --oneline -1

## Install

In [ ]:
!pip install -q -r requirements.txt
if RUN_E4:  # PBT runtime (usually preinstalled on Colab); unused PBT imports are stubbed in qr/pbt_e4.py
    !pip install -q transformers accelerate safetensors joblib

## Mount Drive

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
BASE = "/content/drive/MyDrive/btp_quick_results"
DATA, OUT = f"{BASE}/data", f"{BASE}/outputs"
os.makedirs(OUT, exist_ok=True)
if RUN_E4:
    from google.colab import userdata
    try:
        os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
    except Exception as e:
        print("no HF_TOKEN secret, E4 will skip:", e)

## Download data (SNL, CALCE, HNEI ≈ 0.25 GB; labels; PBT prompt embeddings 6 MB; PBT checkpoint 208 MB if RUN_E4)

In [ ]:
!python run_all.py --data "$DATA" --out "$OUT" --download --steps "" {"--use-hust" if USE_HUST else ""} {"--e4" if RUN_E4 else ""} {"--allow-gate-fail" if ALLOW_GATE_FAIL else ""}
!du -sh "$DATA"

## E0: pipeline + validation gate

In [ ]:
!python run_all.py --data "$DATA" --out "$OUT" --steps e0 {"--use-hust" if USE_HUST else ""} {"--e4" if RUN_E4 else ""} {"--allow-gate-fail" if ALLOW_GATE_FAIL else ""}

## E1: why fraction of life

In [ ]:
!python run_all.py --data "$DATA" --out "$OUT" --steps e1 {"--use-hust" if USE_HUST else ""} {"--e4" if RUN_E4 else ""} {"--allow-gate-fail" if ALLOW_GATE_FAIL else ""}

## E2: cross-chemistry gap (single head, LightGBM)

In [ ]:
!python run_all.py --data "$DATA" --out "$OUT" --steps e2 {"--use-hust" if USE_HUST else ""} {"--e4" if RUN_E4 else ""} {"--allow-gate-fail" if ALLOW_GATE_FAIL else ""}

## E3: mini Track B (MoE hard mask vs additive bias)
`--force e3` reruns E3 even if `e3_raw.csv` exists on Drive. Drop it to reuse cached E3 results.

In [ ]:
!python run_all.py --data "$DATA" --out "$OUT" --steps e3 --force e3 {"--use-hust" if USE_HUST else ""} {"--e4" if RUN_E4 else ""} {"--allow-gate-fail" if ALLOW_GATE_FAIL else ""}

## Check Llama access (informational)
E4 does **not** need Llama: PBT's tokenizer is never used and its prompt embeddings are cached (Zenodo). This only reports whether your `HF_TOKEN` secret can read the tokenizer.

In [ ]:
tok = None
try:
    from google.colab import userdata
    tok = userdata.get("HF_TOKEN")
except Exception as e:
    print("HF_TOKEN secret not available:", type(e).__name__, e)
if tok:
    from huggingface_hub import hf_hub_download
    try:  # one small JSON file, never the weights
        hf_hub_download("meta-llama/Llama-3.1-8B-Instruct", "tokenizer_config.json", token=tok, cache_dir="/content/hf_check")
        print("Llama tokenizer access: OK")
    except Exception as e:
        print("Llama tokenizer access: FAILED:", type(e).__name__, str(e)[:300])
print("E4 runs either way (tokenizer not needed).")

## E4 (RUN_E4): released PBT on our LOCO protocol
E4a zero-shot + E4b PBT fine-tuned on the same few-shot cells as E2. `--force e4` reruns even if E4 outputs exist. Budget ~55 min on a T4; E4b drops seeds 1-2 if the budget would be exceeded (the report says so).

In [ ]:
!python run_all.py --data "$DATA" --out "$OUT" --steps e4 --force e4 {"--use-hust" if USE_HUST else ""} {"--e4" if RUN_E4 else ""} {"--allow-gate-fail" if ALLOW_GATE_FAIL else ""}

## Report

In [ ]:
!python run_all.py --data "$DATA" --out "$OUT" --steps report {"--use-hust" if USE_HUST else ""} {"--e4" if RUN_E4 else ""} {"--allow-gate-fail" if ALLOW_GATE_FAIL else ""}
!mkdir -p outputs && cp -r "$OUT"/. outputs/
from IPython.display import Markdown, display
display(Markdown(open(f"{OUT}/REPORT.md").read()))